# Lab W2 - Citra, CNN, dan Smoke Test Tiga Level

**Modul:** Minggu 2 | **Prasyarat:** Bab 02 §1.5, §2.3, §2.5 + Lab W1 selesai

## Tujuan

1. Memahami representasi citra sebagai tensor: (H,W) kemudian (C,H,W) kemudian (B,C,H,W).
2. Memahami cara kerja `Conv2d`: filter sliding, output shape, dan parameter sharing.
3. Membangun `SimpleCNN` lapis per lapis sebagai `nn.Module`.
4. Menerapkan smoke test tiga level sebelum training penuh.
5. Melatih dan mendiagnosis training `SimpleCNN` pada CIFAR-10.

## Pre-flight Checklist

- [ ] Bab 02 sudah dibaca, khususnya §1.5, §2.3, §2.5.
- [ ] Lab W1 (`lab_w1_tabular_heads.ipynb`) sudah selesai.
- [ ] PyTorch terinstal: `pip install torch torchvision`.

## Deliverables

1. Level 3 smoke test lulus - loss < 0.1 pada iterasi ke-100.
2. Training 10 epoch selesai tanpa error.
3. Plot loss + accuracy dihasilkan.
4. Confusion matrix divisualisasikan.
5. Tabel smoke test error terisi (§4 Latihan).
6. Tiga refleksi dijawab.

**Estimasi waktu:** 3-4 jam (download CIFAR-10 pertama kali + training sekitar 10 menit CPU).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch : {torch.__version__}')
print(f'Device  : {device}')

## §1 - Citra sebagai Tensor

Setiap gambar digital tersimpan sebagai susunan angka. Gambar grayscale berukuran 28x28 piksel tersimpan sebagai matriks dengan shape `(H, W)` = `(28, 28)`. Gambar berwarna (RGB) menambahkan dimensi ketiga untuk channel warna, sehingga shape-nya menjadi `(C, H, W)` = `(3, 32, 32)` untuk satu gambar RGB berukuran 32x32.

PyTorch menggunakan konvensi **channel-first**: urutan dimensi adalah `(C, H, W)`, bukan `(H, W, C)` seperti yang dipakai Matplotlib. Saat kita menggabungkan banyak gambar menjadi satu batch untuk training, dimensi batch (B) ditambahkan di depan, sehingga shape batch menjadi `(B, C, H, W)`.

Tiga sel berikut menunjukkan transisi dari satu gambar ke batch secara bertahap.

In [ ]:
# Download CIFAR-10 ke folder ../data (sekitar 170 MB, hanya pertama kali)
transform_raw = transforms.ToTensor()
cifar_raw = datasets.CIFAR10(root='../data', train=True, download=True, transform=transform_raw)
print(f'Jumlah gambar (train) : {len(cifar_raw):,}')
print(f'Kelas                 : {cifar_raw.classes}')

In [ ]:
img_tensor, label = cifar_raw[0]
print(f'Shape satu gambar : {img_tensor.shape}  # (C, H, W)')
print(f'Label             : {cifar_raw.classes[label]}')
print(f'Nilai piksel      : min={img_tensor.min():.3f}, max={img_tensor.max():.3f}')
print()
print('ToTensor() mengubah nilai piksel dari [0, 255] ke [0.0, 1.0].')

# Matplotlib butuh (H, W, C), bukan (C, H, W) - pakai .permute()
img_np = img_tensor.permute(1, 2, 0).numpy()
plt.figure(figsize=(3, 3))
plt.imshow(img_np)
plt.title(f'Label: {cifar_raw.classes[label]}')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Visualisasi 3 channel secara terpisah
fig, axes = plt.subplots(1, 4, figsize=(10, 2.5))
channel_names = ['Merah (R)', 'Hijau (G)', 'Biru (B)']
cmaps = ['Reds_r', 'Greens_r', 'Blues_r']

axes[0].imshow(img_np)
axes[0].set_title('RGB (gabungan)')
axes[0].axis('off')

for i, (name, cmap) in enumerate(zip(channel_names, cmaps)):
    axes[i + 1].imshow(img_tensor[i].numpy(), cmap=cmap)
    axes[i + 1].set_title(name)
    axes[i + 1].axis('off')

plt.tight_layout()
plt.show()

print(f'Channel R - mean piksel: {img_tensor[0].mean():.3f}')
print(f'Channel G - mean piksel: {img_tensor[1].mean():.3f}')
print(f'Channel B - mean piksel: {img_tensor[2].mean():.3f}')
print('Piksel cerah di satu channel berarti warna itu mendominasi di posisi tersebut.')

In [ ]:
# Dimensi batch (B) ditambahkan saat menggabungkan beberapa gambar
imgs = torch.stack([cifar_raw[i][0] for i in range(4)])
print(f'Shape 4 gambar terpisah : 4 x {tuple(cifar_raw[0][0].shape)}')
print(f'Shape setelah stack     : {tuple(imgs.shape)}  # (B, C, H, W)')
print()
print('Konvensi PyTorch untuk batch citra: (B, C, H, W)')
print(f'  B = {imgs.shape[0]}  -> jumlah gambar dalam satu batch')
print(f'  C = {imgs.shape[1]}  -> jumlah channel (3 untuk RGB)')
print(f'  H = {imgs.shape[2]}  -> tinggi gambar (height)')
print(f'  W = {imgs.shape[3]}  -> lebar gambar (width)')

## §2 - Memahami Conv2d

`nn.Conv2d` menerapkan satu atau lebih filter pada tensor input. Setiap filter berukuran `(kH, kW)` dan bergerak (slide) di sepanjang dimensi H dan W dari input. Pada setiap posisi, filter menghitung dot product dengan patch lokal input berukuran sama, menghasilkan satu nilai skalar output.

Dua properti kunci Conv2d:

1. **Parameter sharing** - filter yang sama digunakan di semua posisi spasial. Ini berarti filter yang mendeteksi satu pola (misalnya tepi diagonal) akan mendeteksi pola tersebut di mana pun ia muncul dalam gambar.
2. **Translation equivariance** - jika suatu pola digeser posisinya dalam gambar, aktivasi output juga bergeser dengan cara yang sama. CNN mengeksploitasi properti ini untuk mendeteksi pola tanpa perlu melatih ulang untuk setiap posisi.

Sel berikut menunjukkan perhitungan Conv2d secara manual sebelum kita menggunakan `nn.Conv2d` PyTorch.

In [ ]:
# Perhitungan manual convolution pada input 5x5 dengan filter 3x3
# (ini persis yang dilakukan nn.Conv2d di balik layar)

x_manual = np.array([
    [1, 1, 1, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 1, 1, 1],
    [0, 0, 1, 1, 0],
    [0, 1, 1, 0, 0],
], dtype=float)

# Filter pendeteksi transisi gelap->terang dari kiri ke kanan
kernel = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1],
], dtype=float)

H, W = x_manual.shape
kH, kW = kernel.shape
out_H = H - kH + 1  # tanpa padding: 5 - 3 + 1 = 3
out_W = W - kW + 1
output = np.zeros((out_H, out_W))

for i in range(out_H):
    for j in range(out_W):
        patch = x_manual[i:i + kH, j:j + kW]
        output[i, j] = (patch * kernel).sum()

print('Input (5x5):')
print(x_manual.astype(int))
print()
print('Filter 3x3 (pendeteksi transisi piksel kiri->kanan):')
print(kernel.astype(int))
print()
print(f'Output ({out_H}x{out_W}) - hasil dot product di setiap posisi sliding:')
print(output)
print()
print('Filter yang sama dipakai di semua 3x3 = 9 posisi - itulah parameter sharing.')

### Rumus Output Shape Conv2d

`nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0)`

Output shape dihitung dengan rumus:

```
out_H = (in_H + 2 * padding - kernel_size) // stride + 1
out_W = (in_W + 2 * padding - kernel_size) // stride + 1
```

Tiga konfigurasi yang paling sering dipakai:

| Konfigurasi | Efek pada shape |
|---|---|
| `kernel=3, padding=1, stride=1` | Shape H dan W tidak berubah |
| `kernel=3, padding=1, stride=2` | Shape H dan W berkurang setengah |
| `kernel=3, padding=0, stride=1` | Shape H dan W berkurang 2 |

Contoh: input `(1, 3, 32, 32)`, `kernel=3`, `padding=1`, `stride=1`:
`out_H = (32 + 2*1 - 3) // 1 + 1 = 32` - ukuran dipertahankan.

In [ ]:
# Demo nn.Conv2d dengan berbagai konfigurasi
x_demo = torch.randn(1, 3, 32, 32)

configs = [
    ('kernel=3, pad=0, stride=1', 3, 32, 3, 1, 0),
    ('kernel=3, pad=1, stride=1', 3, 32, 3, 1, 1),
    ('kernel=3, pad=1, stride=2', 3, 32, 3, 2, 1),
    ('kernel=5, pad=2, stride=1', 3, 32, 5, 1, 2),
]

print(f'Input : {tuple(x_demo.shape)}')
print(f'{"Konfigurasi":<35} {"Output shape"}')
print('-' * 55)

for label, in_c, out_c, k, s, p in configs:
    conv = nn.Conv2d(in_c, out_c, k, stride=s, padding=p)
    out = conv(x_demo)
    # Verifikasi dengan rumus
    expected_H = (32 + 2 * p - k) // s + 1
    assert out.shape[2] == expected_H, f'Rumus tidak cocok untuk {label}'
    print(f'{label:<35} {tuple(out.shape)}')

In [ ]:
# LATIHAN: Prediksi output shape menggunakan rumus, lalu jalankan sel untuk verifikasi.
# Ganti None dengan angka hasil perhitunganmu.

# Soal 1: in=(1, 1, 28, 28), kernel=5, padding=0, stride=1
soal1_H = None  # out_H = (28 + 2*0 - 5) // 1 + 1 = ???
soal1_W = None  # out_W = sama dengan out_H

# Soal 2: in=(1, 32, 16, 16), kernel=3, padding=1, stride=2
soal2_H = None  # out_H = (16 + 2*1 - 3) // 2 + 1 = ???
soal2_W = None

# Soal 3: in=(1, 64, 8, 8), kernel=3, padding=0, stride=1
soal3_H = None  # out_H = (8 + 2*0 - 3) // 1 + 1 = ???
soal3_W = None

# --- Jangan ubah kode di bawah ini ---
conv1 = nn.Conv2d(1, 16, kernel_size=5, padding=0, stride=1)
out1 = conv1(torch.randn(1, 1, 28, 28))

conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1, stride=2)
out2 = conv2(torch.randn(1, 32, 16, 16))

conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=0, stride=1)
out3 = conv3(torch.randn(1, 64, 8, 8))

for idx, (soal_H, soal_W, out) in enumerate([
    (soal1_H, soal1_W, out1),
    (soal2_H, soal2_W, out2),
    (soal3_H, soal3_W, out3),
], start=1):
    if soal_H is None:
        print(f'Soal {idx}: Belum diisi.')
    elif soal_H == out.shape[2] and soal_W == out.shape[3]:
        print(f'Soal {idx} BENAR: output {out.shape[2]}x{out.shape[3]}')
    else:
        print(f'Soal {idx} SALAH: jawabanmu {soal_H}x{soal_W}, '
              f'output sebenarnya {out.shape[2]}x{out.shape[3]}')

## §3 - Membangun SimpleCNN Lapis per Lapis

Kita akan membangun `SimpleCNN` yang terdiri dari:

1. **Block conv 1** - `Conv2d(3, 32)` + `BatchNorm2d` + `ReLU` + `MaxPool2d(2)`
2. **Block conv 2** - `Conv2d(32, 64)` + `BatchNorm2d` + `ReLU` + `MaxPool2d(2)`
3. **Classifier** - `AdaptiveAvgPool2d(1)` + `Flatten` + `Dropout(0.3)` + `Linear(64, 10)`

Kita akan membangun dan memeriksa shape output setiap blok sebelum merakit semuanya menjadi satu class.

In [ ]:
# Block conv pertama: Conv2d -> BatchNorm2d -> ReLU -> MaxPool2d
block1 = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=3, padding=1),  # (B,3,H,W) -> (B,32,H,W)
    nn.BatchNorm2d(32),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),                  # (B,32,H,W) -> (B,32,H/2,W/2)
)

x_test = torch.randn(2, 3, 32, 32)
out_b1 = block1(x_test)
print(f'Input        : {tuple(x_test.shape)}')
print(f'Setelah block1: {tuple(out_b1.shape)}')
print()
print(f'MaxPool2d(2) membelah H dan W: 32 -> {out_b1.shape[2]}')
print(f'Conv2d(3->32) menambah channel: 3 -> {out_b1.shape[1]}')

### Kenapa BatchNorm dan MaxPool?

`BatchNorm2d` menormalisasi aktivasi per channel di seluruh batch pada tiap langkah training. Normalisasi ini menstabilkan distribusi aktivasi sehingga gradient lebih konsisten antar layer dan training lebih cepat konvergen. Tanpa BatchNorm, distribusi aktivasi bisa bergeser drastis antar iterasi, memperlambat training secara signifikan.

`MaxPool2d(2)` mengambil nilai maksimum dari setiap region 2x2 piksel, sehingga H dan W berkurang setengah. Dua efek sekaligus: ukuran feature map berkurang (lebih efisien secara komputasi) dan receptive field dari layer berikutnya menjadi lebih luas - satu neuron di layer setelah dua MaxPool sudah "melihat" area 8x8 di input asli.

In [ ]:
# Block conv kedua: channel bertambah lagi, ukuran spatial berkurang lagi
block2 = nn.Sequential(
    nn.Conv2d(32, 64, kernel_size=3, padding=1),  # (B,32,H,W) -> (B,64,H,W)
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),                   # (B,64,H,W) -> (B,64,H/2,W/2)
)

out_b2 = block2(out_b1)
print(f'Input block2  : {tuple(out_b1.shape)}')
print(f'Setelah block2: {tuple(out_b2.shape)}')
print()
print(f'Dua MaxPool -> H dan W: 32 -> 16 -> {out_b2.shape[2]}')
print(f'Dua Conv2d  -> channel: 3 -> 32 -> {out_b2.shape[1]}')

In [ ]:
# Classifier: pool global -> flatten -> dropout -> linear
# AdaptiveAvgPool2d(1) merata-ratakan tiap channel menjadi skalar tunggal
pool  = nn.AdaptiveAvgPool2d(1)
flat  = nn.Flatten()
head  = nn.Sequential(nn.Dropout(0.3), nn.Linear(64, 10))

x_pool  = pool(out_b2)
x_flat  = flat(x_pool)
logits  = head(x_flat)

print(f'Setelah block2          : {tuple(out_b2.shape)}')
print(f'Setelah AdaptiveAvgPool : {tuple(x_pool.shape)}')
print(f'Setelah Flatten         : {tuple(x_flat.shape)}')
print(f'Setelah head (logits)   : {tuple(logits.shape)}')
print()
print('AdaptiveAvgPool2d(1) menghasilkan (B, 64, 1, 1) berapa pun H dan W masukan.')
print('Karena itu Linear menggunakan in_features=64, bukan 64*8*8.')

In [ ]:
# Rakit semua blok menjadi satu nn.Module
class SimpleCNN(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)
        x = self.block2(x)
        x = self.classifier(x)
        return x

In [ ]:
model = SimpleCNN(num_classes=10).to(device)
print(model)
print()
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameter yang dilatih: {n_params:,}')

## §4 - Smoke Test Tiga Level

Smoke test adalah serangkaian pengujian cepat yang dilakukan **sebelum** training penuh untuk memastikan tidak ada bug mendasar. Setiap level menangkap jenis error yang berbeda:

| Level | Apa yang diuji | Error yang ditangkap | Waktu |
|---|---|---|---|
| **1 - Import test** | `model = Arch(); model.eval()` | Typo nama layer, definisi class rusak | < 1 detik |
| **2 - Dummy forward** | Forward pass input acak, cek shape output | Ketidakcocokan shape antar layer | < 5 detik |
| **3 - Overfit 1 batch** | Loop 100 iterasi pada 8 sample, loss harus turun ke < 0.1 | Training loop rusak, loss function salah, gradient tidak mengalir | 1-2 menit |

Jangan lewati level. Jika Level 3 gagal, bug-nya ada di kode - bukan di hyperparameter.

In [ ]:
# Level 1: instantiasi dan set ke eval mode
model_test = SimpleCNN(num_classes=10)
model_test.eval()
print('Level 1 LULUS - SimpleCNN dapat di-instantiasi dan di-set ke eval mode.')

### Level 2 - Dummy Forward Pass

Buat tensor acak dengan shape yang benar `(B, C, H, W)`, jalankan forward pass, lalu verifikasi shape output dengan `assert`. Jika ada ketidakcocokan channel atau ukuran spatial antar layer, error akan muncul di sini - jauh lebih cepat daripada menunggu satu epoch training.

In [ ]:
model_test.train()  # train mode agar BatchNorm berfungsi normal
x_dummy = torch.randn(2, 3, 32, 32)
logits_dummy = model_test(x_dummy)

print(f'Input : {tuple(x_dummy.shape)}')
print(f'Output: {tuple(logits_dummy.shape)}')

assert logits_dummy.shape == (2, 10), \
    f'Shape output salah: {logits_dummy.shape} - harusnya (2, 10)'
print('Level 2 LULUS - forward pass menghasilkan shape yang benar.')

### Level 3 - Overfit Satu Batch Kecil

Level 3 memverifikasi bahwa gradient mengalir dengan benar ke semua parameter dan training loop berfungsi. Caranya: ambil 8 sample dari dataset, jalankan 100 iterasi optimizer pada batch yang **sama** terus-menerus, dan periksa apakah loss turun dari sekitar 2.3 (acak pada 10 kelas) ke < 0.1.

Jika loss tidak turun:
- Cek apakah `loss.backward()` dipanggil.
- Cek apakah loss function sesuai dengan tugas (multiclass = `CrossEntropyLoss`).
- Cek apakah `optimizer.zero_grad()` dipanggil sebelum setiap iterasi.

In [ ]:
torch.manual_seed(42)

# 8 sample dari CIFAR-10 (pakai ToTensor saja, belum augmentasi)
_raw_l3 = datasets.CIFAR10(root='../data', train=True, download=False,
                             transform=transforms.ToTensor())
x_small, y_small = next(iter(DataLoader(Subset(_raw_l3, list(range(8))),
                                         batch_size=8, shuffle=False)))
x_small, y_small = x_small.to(device), y_small.to(device)

model_l3 = SimpleCNN(num_classes=10).to(device)
opt_l3   = optim.Adam(model_l3.parameters(), lr=1e-2)
crit_l3  = nn.CrossEntropyLoss()

print('Iterasi | Loss')
print('-' * 20)
for it in range(100):
    model_l3.train()
    opt_l3.zero_grad()
    loss = crit_l3(model_l3(x_small), y_small)
    loss.backward()
    opt_l3.step()
    if it % 20 == 0:
        print(f'  {it:>3}   | {loss.item():.4f}')

final_loss = loss.item()
print(f'   99   | {final_loss:.4f}')
print()
if final_loss < 0.1:
    print('Level 3 LULUS - model dapat menghafal satu batch kecil.')
elif final_loss < 0.5:
    print('Level 3 PERINGATAN - loss belum cukup rendah. Coba turunkan learning rate.')
else:
    print('Level 3 GAGAL - loss tidak turun. Periksa training loop dan loss function.')

### Latihan: Smoke Test - Level Berapa yang Mendeteksi?

Isi kolom **Level Deteksi** (1, 2, atau 3) dan **Alasan** untuk setiap jenis error berikut.

| Jenis Error | Contoh konkret | Level Deteksi | Alasan |
|---|---|---|---|
| Typo nama layer (`nn.Liner` bukan `nn.Linear`) | `AttributeError: module has no attribute 'Liner'` | ??? | ??? |
| Jumlah channel tidak cocok antar blok (`Conv2d(3,32)` diikuti `Conv2d(16,64)`) | `RuntimeError: Expected input channels 16 got 32` | ??? | ??? |
| Loss function salah untuk multiclass (`nn.MSELoss` bukan `nn.CrossEntropyLoss`) | Loss turun tapi tidak wajar, akurasi stagnan | ??? | ??? |
| Lupa memanggil `loss.backward()` | Parameter tidak berubah sama sekali | ??? | ??? |
| Model dalam `eval()` saat training | BatchNorm stagnan, Dropout tidak aktif | ??? | ??? |
| `nn.Linear(128, 10)` padahal fitur keluar classifier = 64 | `RuntimeError: mat1 and mat2 shapes cannot be multiplied` | ??? | ??? |

## §5 - Training Loop

Struktur loop sama seperti Lab W1: `zero_grad` kemudian forward kemudian loss kemudian `backward` kemudian `step`. Yang berbeda di sini:

- **Augmentasi pada train split** - `RandomHorizontalFlip` dan `RandomCrop` diterapkan hanya pada data train untuk mencegah overfitting. Data val tidak diaugmentasi agar evaluasi stabil.
- **Normalisasi** - nilai piksel dinormalisasi dengan mean dan std CIFAR-10 agar distribusi input mendekati N(0,1).
- **Dua loop per epoch** - satu loop untuk training (dengan `model.train()`), satu loop untuk validasi (dengan `model.eval()` dan `torch.no_grad()`).

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

transform_val = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

train_set = datasets.CIFAR10('../data', train=True,  transform=transform_train, download=False)
val_set   = datasets.CIFAR10('../data', train=False, transform=transform_val,   download=False)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_set,   batch_size=256, shuffle=False, num_workers=0)

print(f'Train : {len(train_set):,} gambar, {len(train_loader)} batch per epoch')
print(f'Val   : {len(val_set):,} gambar, {len(val_loader)} batch per epoch')
print()
print('Augmentasi train : RandomHorizontalFlip + RandomCrop(32, pad=4) + Normalize')
print('Augmentasi val   : Normalize saja (tidak ada augmentasi acak)')

In [ ]:
torch.manual_seed(42)
model     = SimpleCNN(num_classes=10).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model     : SimpleCNN ({n_trainable:,} parameter)')
print(f'Optimizer : Adam, lr=1e-3')
print(f'Loss      : CrossEntropyLoss')

In [ ]:
N_EPOCHS = 10
train_losses, val_losses = [], []
train_accs,   val_accs   = [], []

for epoch in range(1, N_EPOCHS + 1):
    # --- Training ---
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for x_b, y_b in train_loader:
        x_b, y_b = x_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        logits = model(x_b)
        loss   = criterion(logits, y_b)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(y_b)
        correct      += (logits.argmax(dim=1) == y_b).sum().item()
        total        += len(y_b)
    train_loss = running_loss / total
    train_acc  = correct / total

    # --- Validasi ---
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for x_b, y_b in val_loader:
            x_b, y_b = x_b.to(device), y_b.to(device)
            logits    = model(x_b)
            v_loss   += criterion(logits, y_b).item() * len(y_b)
            v_correct += (logits.argmax(dim=1) == y_b).sum().item()
            v_total   += len(y_b)
    val_loss = v_loss / v_total
    val_acc  = v_correct / v_total

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f'Epoch {epoch:>2}/{N_EPOCHS}  '
          f'train_loss={train_loss:.4f}  train_acc={train_acc:.3f}  '
          f'val_loss={val_loss:.4f}  val_acc={val_acc:.3f}')

print()
print('Training selesai.')

In [ ]:
epochs = range(1, N_EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, train_losses, label='Train')
ax1.plot(epochs, val_losses,   label='Val')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss Curve')
ax1.legend()
ax1.grid(True, alpha=0.3)

train_pct = [a * 100 for a in train_accs]
val_pct   = [a * 100 for a in val_accs]
ax2.plot(epochs, train_pct, label='Train')
ax2.plot(epochs, val_pct,   label='Val')
ax2.axhline(55, color='gray', linestyle='--', linewidth=1, label='Batas 55%')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy Curve')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves_w2.png', dpi=120)
plt.show()

gap = train_accs[-1] - val_accs[-1]
print(f'Epoch terakhir:')
print(f'  Train acc : {train_accs[-1]*100:.1f}%')
print(f'  Val acc   : {val_accs[-1]*100:.1f}%')
print(f'  Gap       : {gap*100:.1f}%')
if gap > 0.15:
    print('  Diagnosis : OVERFITTING - gap train-val > 15%')
elif val_accs[-1] < 0.55:
    print('  Diagnosis : UNDERFITTING - val acc < 55% setelah 10 epoch')
else:
    print('  Diagnosis : Wajar - training berlangsung normal')

## §6 - Evaluasi Hasil

### Panduan Membaca Kurva Training

Bab 02 §2.4 mendeskripsikan empat pola utama:

- **Pola A - Sehat**: train loss dan val loss keduanya turun beriringan. Gap kecil dan stabil.
- **Pola B - Overfitting**: train loss terus turun, val loss mulai naik setelah beberapa epoch. Gap melebar.
- **Pola C - Stagnan**: kedua kurva tidak turun signifikan dari awal. Model tidak belajar sama sekali.
- **Pola D - Loss meledak**: loss tiba-tiba melonjak sangat besar atau menjadi NaN.

Hasil yang diharapkan dari SimpleCNN 10 epoch pada CIFAR-10: val accuracy sekitar 60-70%, dengan Pola A atau sedikit B.

In [ ]:
# Evaluasi pada test set (bukan val set)
test_set    = datasets.CIFAR10('../data', train=False, transform=transform_val, download=False)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=0)

model.eval()
all_labels, all_preds = [], []

with torch.no_grad():
    for x_b, y_b in test_loader:
        x_b  = x_b.to(device)
        preds = model(x_b).argmax(dim=1).cpu()
        all_labels.extend(y_b.tolist())
        all_preds.extend(preds.tolist())

all_labels = np.array(all_labels)
all_preds  = np.array(all_preds)
test_acc   = (all_labels == all_preds).mean()
print(f'Test accuracy: {test_acc * 100:.2f}%')

In [ ]:
# Confusion matrix tanpa sklearn - hitung langsung dengan numpy
CLASSES = test_set.classes
N_CLS   = len(CLASSES)

cm = np.zeros((N_CLS, N_CLS), dtype=int)
for t, p in zip(all_labels, all_preds):
    cm[t, p] += 1

# Normalisasi per baris (akurasi per kelas true)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax.set_xticks(range(N_CLS))
ax.set_yticks(range(N_CLS))
ax.set_xticklabels(CLASSES, rotation=45, ha='right')
ax.set_yticklabels(CLASSES)
ax.set_xlabel('Prediksi')
ax.set_ylabel('Label Benar')
ax.set_title('Confusion Matrix (dinormalisasi per baris)')

for i in range(N_CLS):
    for j in range(N_CLS):
        val   = cm_norm[i, j]
        color = 'white' if val > 0.5 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8, color=color)

plt.tight_layout()
plt.savefig('confusion_matrix_w2.png', dpi=120)
plt.show()

print('Akurasi per kelas:')
for i, cls in enumerate(CLASSES):
    print(f'  {cls:<12}: {cm_norm[i, i] * 100:.1f}%')

In [ ]:
# Tiga pasang kelas yang paling sering tertukar
confusions = []
for i in range(N_CLS):
    for j in range(N_CLS):
        if i != j:
            confusions.append((cm[i, j], CLASSES[i], CLASSES[j]))

confusions.sort(reverse=True)

print('Tiga pasang kelas paling sering tertukar:')
print(f'{"True -> Prediksi":<28} {"Jumlah"}')
print('-' * 38)
for count, true_cls, pred_cls in confusions[:3]:
    print(f'  {true_cls} -> {pred_cls:<22} {count}')

## §7 - Refleksi

Jawab tiga pertanyaan berikut di sel selanjutnya. Tiap jawaban cukup 2-4 kalimat.

**Pertanyaan 1 - Confusion matrix:**
Pasang kelas mana yang paling sering tertukar dalam confusion matrix? Apa karakteristik visual kedua kelas itu yang mungkin menjadi penyebabnya?

**Pertanyaan 2 - Level 3 smoke test:**
Loss Level 3 kamu turun dari berapa ke berapa pada iterasi ke-100? Jika loss tidak mencapai < 0.1, sebutkan satu kemungkinan penyebabnya dan bagaimana cara memverifikasinya.

**Pertanyaan 3 - Keputusan desain:**
Pilih satu keputusan desain di `SimpleCNN` dan jelaskan trade-off-nya:
- (a) `padding=1` di setiap `Conv2d` - kenapa bukan `padding=0`?
- (b) `BatchNorm2d` sebelum `ReLU` - apa dampaknya jika urutannya dibalik?
- (c) `AdaptiveAvgPool2d(1)` di classifier - mengapa bukan `Flatten` langsung?
- (d) `Dropout(0.3)` hanya di classifier - kenapa tidak di dalam conv block?

### Template Jawaban

**Jawaban 1:**

*(Tulis di sini)*

---

**Jawaban 2:**

*(Tulis di sini)*

---

**Jawaban 3 (pilih salah satu: a/b/c/d):**

*(Tulis di sini)*

---

### Self-Check

- [ ] Level 1, 2, dan 3 smoke test sudah dijalankan dan hasilnya tercatat.
- [ ] Level 3: loss turun ke < 0.1 (atau alasan kegagalan sudah diidentifikasi).
- [ ] Training 10 epoch selesai tanpa error.
- [ ] Plot `training_curves_w2.png` tersimpan.
- [ ] Plot `confusion_matrix_w2.png` tersimpan.
- [ ] Tabel latihan smoke test (§4) terisi untuk semua 6 baris.
- [ ] Tiga refleksi dijawab dengan kalimat lengkap.